In [1]:
import os
import json
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor

import rasterio
from rasterio.transform import from_origin
import gc

In [ ]:
maize_env = pd.read_csv('../../dataset/region prediction/maize_soil_all.csv')
wheat_env = pd.read_csv('../../dataset/region prediction/wheat_soil_all.csv')
drop_cols = ['crop_type']
maize_env = maize_env.drop(
    columns=drop_cols,
    errors='ignore'
)

wheat_env = wheat_env.drop(
    columns=drop_cols,
    errors='ignore'
)

In [3]:
print(maize_env.shape)
print(wheat_env.shape)

(1698155, 8)
(708430, 8)


In [ ]:
model = CatBoostRegressor()
model.load_model("../../Model/catboost_china_prediction_no_src.cbm")

In [ ]:
with open("../../Model/features_china_prediction_no_src.json", "r") as f:
    feature_names = json.load(f)

print("\nTotal features:", len(feature_names))


Total features: 43


In [5]:
crop_traits = {
    "maize": {
        "Family": "Poaceae",
        "Genus": "Zea",
        "Order": "Poales",
        "monocot": 1,
        "woody": 0,
        "herb": 1,
        "crop": 1,
        "vegetable": 0,
        "legume": 0,
        "grass": 1,
        "perennial": 0,
        "edible": 1,
        "logED": 2.176091259
    },

    "wheat": {
        "Family": "Poaceae",
        "Genus": "Triticum",
        "Order": "Poales",
        "monocot": 1,
        "woody": 0,
        "herb": 1,
        "crop": 1,
        "vegetable": 0,
        "legume": 0,
        "grass": 1,
        "perennial": 0,
        "edible": 1,
        "logED": 2.380211242
    }
}

In [ ]:
pfas_table = pd.read_excel("../../dataset/PFAS.xlsx")
pfas_table['logIpc'] = np.log10(pfas_table['Ipc'])
print(pfas_table.shape)
print(pfas_table.head())

(41, 21)
  PFAS_name                                         Raw_SMILES  \
0      PFBA                     C(=O)(C(C(C(F)(F)F)(F)F)(F)F)O   
1     PFPeA              C(=O)(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)O   
2     PFHxA       C(=O)(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)(F)F)O   
3     PFHpA  C(=O)(C(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)(F)F)(...   
4      PFOA  C(=O)(C(C(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)(F)F...   

   C-F chain length           Ipc  BCUT2D_MRLOW  BCUT2D_MWLOW        SPS  \
0                 3    260.658616     -0.347050     10.147392  14.384615   
1                 4    939.726831     -0.389910     10.044436  15.062500   
2                 5   3314.446802     -0.417659      9.980602  15.526316   
3                 6  11530.539061     -0.436369      9.938399  15.863636   
4                 7  39741.897337     -0.449718      9.909071  16.120000   

   FractionCSP3  VSA_EState7  MaxAbsEStateIndex  ...    Kappa3  HallKierAlpha  \
0      0.750000    -6.602292          11.750579  ...  1.

In [7]:
# ==========================================================
# Remove unnecessary columns
# ==========================================================

drop_cols = ["Raw_SMILES","C-F chain length",'Ipc']

pfas_table = pfas_table.drop(

    columns=drop_cols,

    errors="ignore"
)

In [8]:
# ==========================================================
# Convert descriptor columns to numeric
# ==========================================================

for col in pfas_table.columns:

    if col != "PFAS_name":

        pfas_table[col] = pd.to_numeric(

            pfas_table[col],

            errors="coerce"
        )

In [9]:
# ==========================================================
# Build PFAS_DICT
# ==========================================================

PFAS_DICT = {}

for _, row in pfas_table.iterrows():

    pfas_name = row["PFAS_name"]

    descriptor_dict = row.drop(

        labels=["PFAS_name"]

    ).to_dict()

    PFAS_DICT[pfas_name] = descriptor_dict

In [10]:
# ==========================================================
# Check
# ==========================================================

print("\nTotal PFAS:")

print(len(PFAS_DICT))

print("\nPFAS names:")

print(list(PFAS_DICT.keys())[:10])

print("\nExample:")

example_name = list(PFAS_DICT.keys())[0]

print(example_name)

print(PFAS_DICT[example_name])


Total PFAS:
41

PFAS names:
['PFBA', 'PFPeA', 'PFHxA', 'PFHpA', 'PFOA', 'PFNA', 'PFDA', 'PFUnDA', 'PFDoDA', 'PFTrDA']

Example:
PFBA
{'BCUT2D_MRLOW': -0.347050241178715, 'BCUT2D_MWLOW': 10.1473915373546, 'SPS': 14.3846153846154, 'FractionCSP3': 0.75, 'VSA_EState7': -6.60229166666667, 'MaxAbsEStateIndex': 11.7505787037037, 'qed': 0.712441776552488, 'Kappa3': 1.67669845728037, 'HallKierAlpha': -1.02, 'MinAbsEStateIndex': 3.53256944444444, 'MaxPartialCharge': 0.460094687009651, 'MinAbsPartialCharge': 0.460094687009651, 'MinPartialCharge': -0.476625664235965, 'EState_VSA9': 5.10652739484071, 'MaxAbsPartialCharge': 0.476625664235965, 'PEOE_VSA3': 4.79453718407182, 'logIpc': 2.41607208477418}


In [11]:
# ==========================================================
# Build prediction dataset
# ==========================================================

def build_prediction_dataset(
    env_df,
    crop_name,
    pfas_name,
    tissue
):

    df = env_df.copy()

    # ======================================================
    # Add crop traits
    # ======================================================

    for k, v in crop_traits[crop_name].items():

        df[k] = v

    # ======================================================
    # Add PFAS descriptors
    # ======================================================

    for k, v in PFAS_DICT[pfas_name].items():

        df[k] = v

    # ======================================================
    # Add tissue
    # ======================================================

    df["Tissue"] = tissue

    # ======================================================
    # Fill missing columns
    # ======================================================

    for col in feature_names:

        if col not in df.columns:

            if col in [
                "Family",
                "Genus",
                "Order",
                "Tissue"
            ]:

                df[col] = "Unknown"

            else:

                df[col] = 0

    # ======================================================
    # Reorder
    # ======================================================

    df = df[feature_names]

    return df


In [12]:
# ==========================================================
# Fast raster writer
# ==========================================================

def points_to_raster(
    df,
    value_col,
    output_tif,
    resolution=0.05
):

    df = df.copy()

    # ======================================================
    # Quantize coordinates
    # ======================================================

    df["lon"] = (
        np.round(df["lon"] / resolution)
        .astype(np.int32)
        * resolution
    )

    df["lat"] = (
        np.round(df["lat"] / resolution)
        .astype(np.int32)
        * resolution
    )

    # ======================================================
    # Extent
    # ======================================================

    xmin = df["lon"].min()
    xmax = df["lon"].max()

    ymin = df["lat"].min()
    ymax = df["lat"].max()

    # ======================================================
    # Raster size
    # ======================================================

    ncols = int(
        round((xmax - xmin) / resolution)
    ) + 1

    nrows = int(
        round((ymax - ymin) / resolution)
    ) + 1

    print("\nRaster size:")
    print(nrows, ncols)

    # ======================================================
    # Empty raster
    # ======================================================

    raster = np.full(
        (nrows, ncols),
        np.nan,
        dtype=np.float32
    )

    # ======================================================
    # Row / col
    # ======================================================

    cols = (
        (df["lon"] - xmin)
        / resolution
    ).round().astype(np.int32)

    rows = (
        (ymax - df["lat"])
        / resolution
    ).round().astype(np.int32)

    # ======================================================
    # Fill raster
    # ======================================================

    raster[
        rows,
        cols
    ] = df[value_col].values

    # ======================================================
    # Transform
    # ======================================================

    transform = from_origin(

        xmin - resolution / 2,

        ymax + resolution / 2,

        resolution,

        resolution
    )
    # ======================================================
    # Save tif
    # ======================================================

    with rasterio.open(

        output_tif,

        'w',

        driver='GTiff',

        height=nrows,

        width=ncols,

        count=1,

        dtype='float32',

        crs='EPSG:4326',

        transform=transform,

        nodata=np.nan,

        compress='lzw'

    ) as dst:

        dst.write(raster, 1)

    print("\nGeoTIFF saved:")
    print(output_tif)

PFAS	Group
PFBA	C4
PFPeA	C5
PFHxA	C6
PFHpA	C7
PFOA	C8
PFNA	C9
PFDA	C10
PFUnDA	C11
PFDoDA	C12
PFTrDA	C13
PFTeDA  C14

    "PFBA",
    "PFPeA",
    "PFHxA",
    "PFHpA",
    "PFOA",
    "PFNA",
    "PFDA",
    "PFUnDA",
    "PFDoDA",
    "PFTrDA",
    "PFTeDA",
    "PFBS",
    "PFPeS",
    "PFHxS",
    "PFHpS",
    "PFOS",
    "PFNS",
    "PFDS"

In [ ]:
# ==========================================================
# Batch spatial prediction
# ==========================================================

PFAS_LIST = [
    "PFBA",
    "PFPeA",
    "PFHxA",
    "PFHpA",
    "PFOA",
    "PFNA",
    "PFDA",
    "PFUnDA",
    "PFDoDA",
    "PFTrDA",
    "PFTeDA",
    "PFBS",
    "PFPeS",
    "PFHxS",
    "PFHpS",
    "PFOS",
    "PFNS",
    "PFDS",
    "GenX"
]

CROP_LIST = [
    "wheat",
    "maize"
]

TISSUE_LIST = [
    "root",
    "fruit"
]

In [ ]:
# ==========================================================
# Loop
# ==========================================================

for crop_name in CROP_LIST:

    # ======================================================
    # Select environmental data
    # ======================================================

    if crop_name == "wheat":

        env_df = wheat_env

    elif crop_name == "maize":

        env_df = maize_env

    # ======================================================
    # PFAS loop
    # ======================================================

    for pfas_name in PFAS_LIST:

        # ==================================================
        # Tissue loop
        # ==================================================

        for tissue in TISSUE_LIST:

            print("\n=================================")
            print(
                crop_name,
                pfas_name,
                tissue
            )
            print("=================================")

            # ==============================================
            # Build prediction matrix
            # ==============================================

            pred_df = build_prediction_dataset(
                env_df=env_df,
                crop_name=crop_name,
                pfas_name=pfas_name,
                tissue=tissue
            )

            # ==============================================
            # Prediction
            # ==============================================

            pred = model.predict(pred_df)

            pred = pred.astype(np.float32)

            # ==============================================
            # Compact results
            # ==============================================

            result_df = pd.DataFrame({

                "lon": env_df["lon"].astype(np.float32),

                "lat": env_df["lat"].astype(np.float32),

                "lnBCF_pred": pred

            })

            # ==============================================
            # Pickle path
            # ==============================================

            pickle_path = (

                f"../../result/"
                f"{pfas_name}_"
                f"{crop_name}_"
                f"{tissue}.pkl"
            )

            result_df.to_pickle(
                pickle_path
            )

            print("\nPickle saved:")
            print(pickle_path)

            # ==============================================
            # GeoTIFF path
            # ==============================================

            tif_path = (

                f"../../result/tif/"
                f"{pfas_name}_"
                f"{crop_name}_"
                f"{tissue}.tif"
            )

            # ==============================================
            # Save raster
            # ==============================================

            points_to_raster(
                result_df,
                value_col='lnBCF_pred',
                output_tif=tif_path,
                resolution=0.05
            )

            # ==============================================
            # Memory cleanup
            # ==============================================

            del pred_df
            del pred
            del result_df

            gc.collect()

print("\nAll scenarios finished.")


wheat PFBA root

Pickle saved:
../result/PFBA_wheat_root.pkl

Raster size:
702 1206

GeoTIFF saved:
../result/tif/PFBA_wheat_root.tif

wheat PFBA fruit

Pickle saved:
../result/PFBA_wheat_fruit.pkl

Raster size:
702 1206

GeoTIFF saved:
../result/tif/PFBA_wheat_fruit.tif

wheat PFPeA root

Pickle saved:
../result/PFPeA_wheat_root.pkl

Raster size:
702 1206

GeoTIFF saved:
../result/tif/PFPeA_wheat_root.tif

wheat PFPeA fruit

Pickle saved:
../result/PFPeA_wheat_fruit.pkl

Raster size:
702 1206

GeoTIFF saved:
../result/tif/PFPeA_wheat_fruit.tif

wheat PFHxA root

Pickle saved:
../result/PFHxA_wheat_root.pkl

Raster size:
702 1206

GeoTIFF saved:
../result/tif/PFHxA_wheat_root.tif

wheat PFHxA fruit

Pickle saved:
../result/PFHxA_wheat_fruit.pkl

Raster size:
702 1206

GeoTIFF saved:
../result/tif/PFHxA_wheat_fruit.tif

wheat PFHpA root

Pickle saved:
../result/PFHpA_wheat_root.pkl

Raster size:
702 1206

GeoTIFF saved:
../result/tif/PFHpA_wheat_root.tif

wheat PFHpA fruit

Pickle save

In [ ]:
# ==========================================================
# Chain-length grouped lnBCF
# ==========================================================

CHAIN_GROUPS = {

    "short": [
        "PFBA",
        "PFPeA",
        "PFHxA",
        "PFBS",
        "PFPeS",
        "PFHxS"
    ],

    "medium": [
        "PFHpA",
        "PFOA",
        "PFNA",
        "PFHpS",
        "PFOS",
        "PFNS"
    ],

    "long": [
        "PFDA",
        "PFUnDA",
        "PFDoDA",
        "PFTrDA",
        "PFTeDA",
        "PFDS"
    ]
}

# ==========================================================
# Loop
# ==========================================================

for crop_name in CROP_LIST:

    for tissue in TISSUE_LIST:

        for group_name, pfas_list in CHAIN_GROUPS.items():

            print("\n=================================")
            print(
                "Chain group:",
                crop_name,
                tissue,
                group_name
            )
            print("=================================")

            # ==============================================
            # Store lnBCF arrays
            # ==============================================

            lnbcf_stack = []

            coord_df = None

            # ==============================================
            # PFAS loop
            # ==============================================

            for pfas_name in pfas_list:

                pickle_path = (

                    f"../../result/"
                    f"{pfas_name}_"
                    f"{crop_name}_"
                    f"{tissue}.pkl"
                )

                print("\nLoading:")
                print(pickle_path)

                df = pd.read_pickle(
                    pickle_path
                )

                # ==========================================
                # Save coordinates once
                # ==========================================

                if coord_df is None:

                    coord_df = df[
                        ["lon", "lat"]
                    ].copy()

                # ==========================================
                # Stack lnBCF
                # ==========================================

                lnbcf_stack.append(

                    df["lnBCF_pred"].values
                )

                del df

                gc.collect()

            # ==============================================
            # Convert to numpy stack
            # ==============================================

            lnbcf_stack = np.vstack(
                lnbcf_stack
            )

            print("\nStack shape:")
            print(lnbcf_stack.shape)

            # ==============================================
            # Mean lnBCF
            # ==============================================

            mean_lnbcf = np.mean(

                lnbcf_stack,

                axis=0

            ).astype(np.float32)

            # ==============================================
            # Result dataframe
            # ==============================================

            result_df = coord_df.copy()

            result_df["lnBCF_pred"] = mean_lnbcf

            # ==============================================
            # Save pickle
            # ==============================================

            out_pkl = (

                f"../../result/all/"
                f"{group_name}_"
                f"{crop_name}_"
                f"{tissue}.pkl"
            )

            result_df.to_pickle(
                out_pkl
            )

            print("\nGrouped pickle saved:")
            print(out_pkl)

            # ==============================================
            # Save tif
            # ==============================================

            out_tif = (

                f"../../result/all/tif/"
                f"{group_name}_"
                f"{crop_name}_"
                f"{tissue}.tif"
            )

            points_to_raster(

                result_df,

                value_col="lnBCF_pred",

                output_tif=out_tif,

                resolution=0.05
            )

            print("\nGrouped tif saved:")
            print(out_tif)

            # ==============================================
            # Summary statistics
            # ==============================================

            print("\nSummary:")

            print(

                result_df["lnBCF_pred"]

                .describe()
            )

            # ==============================================
            # Cleanup
            # ==============================================

            del lnbcf_stack
            del mean_lnbcf
            del result_df

            gc.collect()

print("\n=================================")
print("All chain-group analyses finished.")
print("=================================")


Chain group: wheat root short

Loading:
../result/PFBA_wheat_root.pkl

Loading:
../result/PFPeA_wheat_root.pkl

Loading:
../result/PFHxA_wheat_root.pkl

Loading:
../result/PFBS_wheat_root.pkl

Loading:
../result/PFPeS_wheat_root.pkl

Loading:
../result/PFHxS_wheat_root.pkl

Stack shape:
(6, 708430)

Grouped pickle saved:
../result/all/short_wheat_root.pkl

Raster size:
702 1206

GeoTIFF saved:
../result/all/tif/short_wheat_root.tif

Grouped tif saved:
../result/all/tif/short_wheat_root.tif

Summary:
count    708430.000000
mean          0.882502
std           0.122794
min           0.365362
25%           0.798418
50%           0.884043
75%           0.954234
max           1.715179
Name: lnBCF_pred, dtype: float64

Chain group: wheat root medium

Loading:
../result/PFHpA_wheat_root.pkl

Loading:
../result/PFOA_wheat_root.pkl

Loading:
../result/PFNA_wheat_root.pkl

Loading:
../result/PFHpS_wheat_root.pkl

Loading:
../result/PFOS_wheat_root.pkl

Loading:
../result/PFNS_wheat_root.pkl

St

In [ ]:
# ==========================================================
# Chain-group TF analysis
# ==========================================================

summary_results = []

# ==========================================================
# Loop
# ==========================================================

for crop_name in CROP_LIST:

    for group_name in CHAIN_GROUPS.keys():

        print("\n=================================")
        print(
            "Grouped TF analysis:",
            crop_name,
            group_name
        )
        print("=================================")

        # ==================================================
        # Load root
        # ==================================================

        root_path = (

            f"../../result/all/"
            f"{group_name}_"
            f"{crop_name}_"
            f"root.pkl"
        )

        root_df = pd.read_pickle(
            root_path
        )

        # ==================================================
        # Load fruit
        # ==================================================

        fruit_path = (

            f"../../result/all/"
            f"{group_name}_"
            f"{crop_name}_"
            f"fruit.pkl"
        )

        fruit_df = pd.read_pickle(
            fruit_path
        )

        # ==================================================
        # Direct combine
        # ==================================================

        tf_df = pd.DataFrame({

            "lon":
                root_df["lon"].values,

            "lat":
                root_df["lat"].values,

            "lnBCF_root":
                root_df["lnBCF_pred"].values,

            "lnBCF_fruit":
                fruit_df["lnBCF_pred"].values
        })

        # ==================================================
        # log10 TF
        # ==================================================

        tf_df["logTF"] = (

            tf_df["lnBCF_fruit"]

            -

            tf_df["lnBCF_root"]

        ) * np.log10(np.e)

        tf_df["logTF"] = (
            tf_df["logTF"]
            .astype(np.float32)
        )

        # ==================================================
        # Remove outliers
        # ==================================================

        q01 = tf_df["logTF"].quantile(0.01)

        q99 = tf_df["logTF"].quantile(0.99)

        tf_df["logTF"] = tf_df["logTF"].clip(
            q01,
            q99
        )

        # ==================================================
        # Summary
        # ==================================================

        summary_results.append({

            "Crop": crop_name,

            "Chain_group": group_name,

            "Mean_logTF":
                tf_df["logTF"].mean(),

            "Median_logTF":
                tf_df["logTF"].median(),

            "P95_logTF":
                tf_df["logTF"].quantile(0.95),

            "Max_logTF":
                tf_df["logTF"].max()
        })

        # ==================================================
        # Save pickle
        # ==================================================

        tf_pickle_path = (

            f"../../result/TF/all/"
            f"{group_name}_"
            f"{crop_name}_TF.pkl"
        )

        tf_df.to_pickle(
            tf_pickle_path
        )

        print("\nTF pickle saved:")
        print(tf_pickle_path)

        # ==================================================
        # Save tif
        # ==================================================

        tf_tif_path = (

            f"../../result/TF/all/tif/"
            f"{group_name}_"
            f"{crop_name}_TF.tif"
        )

        points_to_raster(

            tf_df,

            value_col="logTF",

            output_tif=tf_tif_path,

            resolution=0.05
        )

        print("\nTF tif saved:")
        print(tf_tif_path)

        # ==================================================
        # Cleanup
        # ==================================================

        del root_df
        del fruit_df
        del tf_df

        gc.collect()

# ==========================================================
# Save summary
# ==========================================================

summary_df = pd.DataFrame(
    summary_results
)

summary_df.to_excel(

    "../../result/TF/all/TF_summary_group.xlsx",

    index=False
)

print("\n=================================")
print("All grouped TF analyses finished.")
print("=================================")


Grouped TF analysis: wheat short

TF pickle saved:
../result/TF/all/short_wheat_TF.pkl

Raster size:
702 1206

GeoTIFF saved:
../result/TF/all/tif/short_wheat_TF.tif

TF tif saved:
../result/TF/all/tif/short_wheat_TF.tif

Grouped TF analysis: wheat medium

TF pickle saved:
../result/TF/all/medium_wheat_TF.pkl

Raster size:
702 1206

GeoTIFF saved:
../result/TF/all/tif/medium_wheat_TF.tif

TF tif saved:
../result/TF/all/tif/medium_wheat_TF.tif

Grouped TF analysis: wheat long

TF pickle saved:
../result/TF/all/long_wheat_TF.pkl

Raster size:
702 1206

GeoTIFF saved:
../result/TF/all/tif/long_wheat_TF.tif

TF tif saved:
../result/TF/all/tif/long_wheat_TF.tif

Grouped TF analysis: maize short

TF pickle saved:
../result/TF/all/short_maize_TF.pkl

Raster size:
682 1196

GeoTIFF saved:
../result/TF/all/tif/short_maize_TF.tif

TF tif saved:
../result/TF/all/tif/short_maize_TF.tif

Grouped TF analysis: maize medium

TF pickle saved:
../result/TF/all/medium_maize_TF.pkl

Raster size:
682 1196